In [1]:
# run each time session starts
!pip install ultralytics -q

import torch
from ultralytics import YOLO

print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 7.6 MB/s eta 0:00:0000:0100:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA: True
GPU: Tesla T4


**Data Preprocessing**

In [3]:
#Filter Labels (run once, ~8 min)
# using only 3 Classes instead of 10 
import os
import yaml
from pathlib import Path

KEEP_MAP = {
    0: 0,  # person  → person
    2: 1,  # car     → car
    5: 2,  # bike    → cyclist
    6: 2,  # motor   → cyclist
}

SRC_TRAIN_IMG = '/kaggle/input/datasets/a7madmostafa/bdd100k-yolo/train/images'
SRC_VAL_IMG   = '/kaggle/input/datasets/a7madmostafa/bdd100k-yolo/val/images'
SRC_TRAIN_LBL = '/kaggle/input/datasets/a7madmostafa/bdd100k-yolo/train/labels'
SRC_VAL_LBL   = '/kaggle/input/datasets/a7madmostafa/bdd100k-yolo/val/labels'

DST_TRAIN_IMG = '/kaggle/working/bdd3cls/train/images'
DST_TRAIN_LBL = '/kaggle/working/bdd3cls/train/labels'
DST_VAL_IMG   = '/kaggle/working/bdd3cls/val/images'
DST_VAL_LBL   = '/kaggle/working/bdd3cls/val/labels'

for d in [DST_TRAIN_IMG, DST_TRAIN_LBL, DST_VAL_IMG, DST_VAL_LBL]:
    os.makedirs(d, exist_ok=True)

def filter_and_link(src_lbl_dir, dst_lbl_dir, src_img_dir, dst_img_dir, split):
    src_files = list(Path(src_lbl_dir).glob('*.txt'))
    kept = 0
    for src_path in src_files:
        with open(src_path) as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            orig_cls = int(parts[0])
            if orig_cls in KEEP_MAP:
                new_lines.append(f"{KEEP_MAP[orig_cls]} {' '.join(parts[1:])}\n")
        if new_lines:
            # Write filtered label
            with open(f'{dst_lbl_dir}/{src_path.name}', 'w') as f:
                f.writelines(new_lines)
            # Symlink image
            stem = src_path.stem
            src_img = f'{src_img_dir}/{stem}.jpg'
            dst_img = f'{dst_img_dir}/{stem}.jpg'
            if os.path.exists(src_img) and not os.path.exists(dst_img):
                os.symlink(src_img, dst_img)
            kept += 1
    print(f"{split}: {kept}/{len(src_files)} images kept")
    return kept

n_train = filter_and_link(SRC_TRAIN_LBL, DST_TRAIN_LBL, SRC_TRAIN_IMG, DST_TRAIN_IMG, 'train')
n_val   = filter_and_link(SRC_VAL_LBL,   DST_VAL_LBL,   SRC_VAL_IMG,   DST_VAL_IMG,   'val')

# Write yaml — points entirely to filtered directory
cfg = {
    'path'  : '/kaggle/working/bdd3cls',
    'train' : 'train/images',
    'val'   : 'val/images',
    'nc'    : 3,
    'names' : ['person', 'car', 'cyclist']
}
with open('/kaggle/working/bdd3cls.yaml', 'w') as f:
    yaml.dump(cfg, f)

print(f"\n✅ Done. YAML written.")
print(f"   train: {n_train} images  |  val: {n_val} images")
print(f"   nc=3 : person / car / cyclist")

train: 69447/70000 images kept
val: 9912/10000 images kept

✅ Done. YAML written.
   train: 69447 images  |  val: 9912 images
   nc=3 : person / car / cyclist


In [4]:
# check if filtration is done perfectly
import random
from pathlib import Path

DST_TRAIN_LBL = '/kaggle/working/bdd3cls/train/labels'
DST_TRAIN_IMG = '/kaggle/working/bdd3cls/train/images'

lbl_stems = set(f.stem for f in Path(DST_TRAIN_LBL).glob('*.txt'))
img_stems = set(f.stem for f in Path(DST_TRAIN_IMG).glob('*.jpg'))

print(f"Label files : {len(lbl_stems)}")
print(f"Image links : {len(img_stems)}")
print(f"Matched     : {len(lbl_stems & img_stems)}")
print(f"Labels without image: {len(lbl_stems - img_stems)}")
print(f"Images without label: {len(img_stems - lbl_stems)}")

# Spot check 3 random label files
print("\nSpot check (should only contain classes 0, 1, 2):")
for stem in random.sample(list(lbl_stems), 3):
    with open(f'{DST_TRAIN_LBL}/{stem}.txt') as f:
        lines = f.readlines()
    classes = [int(l.split()[0]) for l in lines if l.strip()]
    print(f"  {stem}: classes={classes}")

Label files : 69447
Image links : 69447
Matched     : 69447
Labels without image: 0
Images without label: 0

Spot check (should only contain classes 0, 1, 2):
  43d65cac-dd45e4d2: classes=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  37f64b1f-f9b09835: classes=[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1]
  06132c7e-0f2f5794: classes=[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]


### After your filtering:
1. **Train** 70,000 - 553 = 69,447
2. **Val**   10,000 - 88  = 9,912
3. **Test**  20,000 - 0   =unused (no labels)
   **Total used = 80,000 - 641 = 79,359**

# Centralized Model training

In [ ]:
# running a centralized model for 3 classes and setting a benchmark of accuracy and latency
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data    = '/kaggle/working/bdd3cls.yaml',
    epochs  = 10,
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    name    = 'central_3cls',
)

# Print the key number
results = model.val(data='/kaggle/working/bdd3cls.yaml', verbose=False)
print(f"\n📊 Centralized baseline mAP@0.5:0.95 = {results.box.map:.4f}")
print(f"   person  : {results.box.maps[0]:.4f}")
print(f"   car     : {results.box.maps[1]:.4f}")
print(f"   cyclist : {results.box.maps[2]:.4f}")

## Centralized Output
**mAP@0.5-0.95 :** 
### 27.8%

### Split the dataset to IID and Non IID

In [5]:
# Data Split (IID and Non IID)
import random
import numpy as np
from pathlib import Path

random.seed(42)
np.random.seed(42)

N_CLIENTS     = 5
DST_TRAIN_LBL = '/kaggle/working/bdd3cls/train/labels'
DST_TRAIN_IMG = '/kaggle/working/bdd3cls/train/images'
FL_DIR        = '/kaggle/working/fl3cls'

valid_stems = [f.stem for f in Path(DST_TRAIN_LBL).glob('*.txt')]
random.shuffle(valid_stems)
print(f"Total valid images: {len(valid_stems)}")

# ── IID split ──
iid_splits = np.array_split(valid_stems, N_CLIENTS)
for i, split in enumerate(iid_splits):
    d = f'{FL_DIR}/iid/client_{i}'
    os.makedirs(f'{d}/images', exist_ok=True)
    os.makedirs(f'{d}/labels', exist_ok=True)
    with open(f'{d}/image_list.txt', 'w') as f:
        for stem in split:
            f.write(stem + '\n')
print(f"IID: {[len(s) for s in iid_splits]}")

# ── Non-IID split ──
def get_dominant(stem):
    lbl = f'{DST_TRAIN_LBL}/{stem}.txt'
    if not os.path.exists(lbl):
        return -1
    with open(lbl) as f:
        lines = f.readlines()
    classes = [int(l.split()[0]) for l in lines if l.strip()]
    return max(set(classes), key=classes.count) if classes else -1

print("Building Non-IID splits...")
buckets = {0: [], 1: [], 2: []}
for stem in valid_stems:
    dc = get_dominant(stem)
    if dc in buckets:
        buckets[dc].append(stem)

print(f"  person-dominant : {len(buckets[0])}")
print(f"  car-dominant    : {len(buckets[1])}")
print(f"  cyclist-dominant: {len(buckets[2])}")

dominant_per_client = [1, 1, 0, 2, 2]
for i, dom_cls in enumerate(dominant_per_client):
    dom_pool   = buckets[dom_cls].copy()
    other_pool = [s for c, imgs in buckets.items()
                  if c != dom_cls for s in imgs]
    random.shuffle(dom_pool)
    random.shuffle(other_pool)
    target  = len(valid_stems) // N_CLIENTS
    n_dom   = min(int(target * 0.7), len(dom_pool))
    n_other = target - n_dom
    client_imgs = dom_pool[:n_dom] + other_pool[:n_other]
    random.shuffle(client_imgs)

    d = f'{FL_DIR}/noniid/client_{i}'
    os.makedirs(f'{d}/images', exist_ok=True)
    os.makedirs(f'{d}/labels', exist_ok=True)
    with open(f'{d}/image_list.txt', 'w') as f:
        for stem in client_imgs:
            f.write(stem + '\n')

print(f"Non-IID: done")
print("✅ FL splits ready!")

Total valid images: 69447
IID: [13890, 13890, 13889, 13889, 13889]
Building Non-IID splits...
  person-dominant : 4387
  car-dominant    : 64917
  cyclist-dominant: 143
Non-IID: done
✅ FL splits ready!


In [6]:
#Federated Learning Implementation

import torch, copy, yaml, time, json, os
import numpy as np
from ultralytics import YOLO

N_CLIENTS    = 5
N_ROUNDS     = 5
LOCAL_EPOCHS = 3
BATCH        = 16
IMGSZ        = 640

DST_TRAIN_LBL = '/kaggle/working/bdd3cls/train/labels'
DST_TRAIN_IMG = '/kaggle/working/bdd3cls/train/images'
SRC_VAL_IMG   = '/kaggle/input/datasets/a7madmostafa/bdd100k-yolo/val/images'
DST_VAL_LBL   = '/kaggle/working/bdd3cls/val/labels'
# BASE_WEIGHTS  = '/kaggle/working/runs/detect/central_3cls/weights/best.pt'
BASE_WEIGHTS  = '/kaggle/input/datasets/mohammedsunain/bdd3cls-weights/central_3cls_best.pt' 
VAL_YAML      = '/kaggle/working/bdd3cls.yaml'
FL_DIR        = '/kaggle/working/fl3cls'
RESULTS_DIR   = '/kaggle/working/fl_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

def setup_client(client_dir):
    with open(f'{client_dir}/image_list.txt') as f:
        stems = [l.strip() for l in f.readlines()]
    for stem in stems:
        dst_img = f'{client_dir}/images/{stem}.jpg'
        dst_lbl = f'{client_dir}/labels/{stem}.txt'
        if not os.path.exists(dst_img):
            src = f'{DST_TRAIN_IMG}/{stem}.jpg'
            if os.path.exists(src):
                os.symlink(src, dst_img)
        if not os.path.exists(dst_lbl):
            src = f'{DST_TRAIN_LBL}/{stem}.txt'
            if os.path.exists(src):
                os.symlink(src, dst_lbl)
    return len(stems)

def write_client_yaml(client_dir):
    cfg = {
        'path'  : client_dir,
        'train' : 'images',
        'val'   : '/kaggle/working/bdd3cls/val/images',
        'nc'    : 3,
        'names' : ['person', 'car', 'cyclist']
    }
    p = f'{client_dir}/data.yaml'
    with open(p, 'w') as f:
        yaml.dump(cfg, f)
    return p

def fedavg(global_w, client_ws, sizes):
    total = sum(sizes)
    avg   = copy.deepcopy(global_w)
    for key in avg:
        if not avg[key].is_floating_point():
            continue
        avg[key] = torch.zeros_like(avg[key], dtype=torch.float32)
        for w, s in zip(client_ws, sizes):
            avg[key] += w[key].float() * (s / total)
    return avg

def fedavgm(prev_m, delta, beta=0.9):
    if prev_m is None:
        return delta
    return {k: (beta * prev_m[k].float() + (1-beta) * delta[k].float()
                if delta[k].is_floating_point() else delta[k])
            for k in delta}

def run_fl_experiment(algorithm, split_type):
    print(f"\n{'='*55}")
    print(f"  {algorithm.upper()} | {split_type.upper()} | 3-class")
    print(f"  R={N_ROUNDS}  E={LOCAL_EPOCHS}  Clients={N_CLIENTS}")
    print(f"{'='*55}\n")

    exp_name  = f"{algorithm}_{split_type}_3cls"
    split_dir = f'{FL_DIR}/{split_type}'

    # Resume if partially done
    out_path = f'{RESULTS_DIR}/{exp_name}.json'
    if os.path.exists(out_path):
        with open(out_path) as f:
            saved = json.load(f)
        if saved.get('complete'):
            print(f"  Already complete — skipping. Final mAP: {saved['final_map']:.4f}")
            return saved

    client_sizes = []
    client_yamls = []
    for i in range(N_CLIENTS):
        client_dir = f'{split_dir}/client_{i}'
        n = setup_client(client_dir)
        client_sizes.append(n)
        client_yamls.append(write_client_yaml(client_dir))
        print(f"  Client {i}: {n} images")

    global_model = YOLO(BASE_WEIGHTS)
    global_state = copy.deepcopy(global_model.model.state_dict())

    round_maps      = []
    per_class_maps  = []
    server_momentum = None
    start_time      = time.time()

    for rnd in range(1, N_ROUNDS + 1):
        print(f"\n--- Round {rnd}/{N_ROUNDS} ---")
        client_weights = []

        for cid in range(N_CLIENTS):
            local_model = YOLO(BASE_WEIGHTS)
            local_model.model.load_state_dict(copy.deepcopy(global_state))
            local_model.train(
                data     = client_yamls[cid],
                epochs   = LOCAL_EPOCHS,
                imgsz    = IMGSZ,
                batch    = BATCH,
                device   = 0,
                name     = f"{exp_name}/r{rnd:02d}_c{cid}",
                exist_ok = True,
                verbose  = False,
                plots    = False,
                lr0      = 0.005 if algorithm == 'fedprox' else 0.01,
            )
            client_weights.append(copy.deepcopy(local_model.model.state_dict()))
            print(f"  ✓ Client {cid}")

        # Aggregation
        if algorithm in ('fedavg', 'fedprox'):
            global_state = fedavg(global_state, client_weights, client_sizes)
        elif algorithm == 'fedavgm':
            avg   = fedavg(global_state, client_weights, client_sizes)
            delta = {k: (avg[k].float() - global_state[k].float()
                         if global_state[k].is_floating_point() else avg[k])
                     for k in global_state}
            server_momentum = fedavgm(server_momentum, delta)
            global_state = {k: (global_state[k].float() + server_momentum[k]
                                if global_state[k].is_floating_point()
                                else global_state[k])
                            for k in global_state}

        # Evaluation
        eval_model = YOLO(BASE_WEIGHTS)
        eval_model.model.load_state_dict(global_state)
        metrics = eval_model.val(
            data=VAL_YAML, imgsz=IMGSZ, batch=32,
            device=0, verbose=False, plots=False,
            name=f"{exp_name}/eval_r{rnd:02d}", exist_ok=True,
        )
        m    = float(metrics.box.map)
        maps = metrics.box.maps
        round_maps.append(m)
        per_class_maps.append({
            'person'  : float(maps[0]),
            'car'     : float(maps[1]),
            'cyclist' : float(maps[2]),
        })
        elapsed = (time.time() - start_time) / 60
        print(f"  📊 R{rnd}: mAP={m:.4f} "
              f"[P={maps[0]:.3f} C={maps[1]:.3f} Cy={maps[2]:.3f}] "
              f"{elapsed:.0f}min")

        # Save after every round so you don't lose progress
        results = {
            'algorithm'     : algorithm,
            'split_type'    : split_type,
            'n_rounds_done' : rnd,
            'round_maps'    : round_maps,
            'per_class_maps': per_class_maps,
            'final_map'     : round_maps[-1],
            'best_map'      : max(round_maps),
            'complete'      : rnd == N_ROUNDS,
        }
        with open(out_path, 'w') as f:
            json.dump(results, f, indent=2)

    print(f"\n✅ {exp_name}: final={round_maps[-1]:.4f}  best={max(round_maps):.4f}")
    return results

print("Done bie")

Done bie


In [16]:
# Downloaded the weights required for 3 classes model
# so that we dont need to run the central model everytime


# import shutil, os

# os.makedirs('/kaggle/working/saved_model', exist_ok=True)
# shutil.copy(
#     '/kaggle/working/runs/detect/central_3cls/weights/best.pt',
#     '/kaggle/working/saved_model/central_3cls_best.pt'
# )
# print("✅ Download central_3cls_best.pt from the output panel NOW")

✅ Download central_3cls_best.pt from the output panel NOW


# FedAvg Model Training

In [ ]:
# SESSION 1 — paste this and let it run overnight
r1 = run_fl_experiment('fedavg', 'iid')
r2 = run_fl_experiment('fedavg', 'noniid')


  FEDAVG | IID | 3-class
  R=5  E=3  Clients=5

  Client 0: 13890 images
  Client 1: 13890 images
  Client 2: 13889 images
  Client 3: 13889 images
  Client 4: 13889 images

--- Round 1/5 ---
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/fl3cls/iid/client_0/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mi

### FedAvg Model Output
1. **noniid - mAP@50-95: 28.6%**
2. **iid - mAP@50-95: 28.2%**


In [ ]:

r3 = run_fl_experiment('fedprox', 'iid')
r4 = run_fl_experiment('fedprox', 'noniid')